# FIN 4000 — Chapter 16: Managing Bond Portfolios
*Duration as interest-rate sensitivity, and why immunization depends on it*

**ANSWER KEY — fully worked, instructor reference**

This notebook is the Python companion to the Chapter 16 Excel template — same data, same formulas, same numbers. Where the Excel workshop has you build a formula in a cell, this notebook has you build it in a line of Python instead.

## Step 1 — Entry Quiz *(3 minutes, individually)*

1. In plain terms, what does duration measure?
2. Why does a bond with a higher coupon rate have a lower duration than an otherwise-identical bond with a lower coupon?
3. What does it mean to “immunize” a bond portfolio against interest-rate risk?

---

In [1]:
import numpy as np

## Step 2 — Worked Example *(10 minutes, instructor-led)*

A 3-year, $1,000 face-value, 6% annual-coupon bond, priced at an 8% yield (price = $948.46, from Chapter 14's method).

In [2]:
import pandas as pd
face, coupon_rate, n, ytm = 1000, 0.06, 3, 0.08
cash_flows = np.array([60, 60, 1060])
periods = np.array([1, 2, 3])

In [3]:
pv = cash_flows / (1 + ytm) ** periods
price = pv.sum()
mac_duration = (periods * pv).sum() / price
pd.DataFrame({'t': periods, 'CF': cash_flows, 'PV': pv.round(2), 't x PV': (periods*pv).round(2)})

,t,CF,PV,t x PV
0,1,60,55.56,55.56
1,2,60,51.44,102.88
2,3,1060,841.46,2524.39


In [4]:
print(f'Price=${price:.2f}  Macaulay Duration={mac_duration:.2f} years')

Price=$948.46  Macaulay Duration=2.83 years


*Σ(t×PV) = 2,682.84; Price = $948.46. Macaulay Duration = 2,682.84 ÷ 948.46 = 2.83 years — shorter than the bond's 3-year maturity, since coupons return some cash before the final year.*

## Step 3 — Pair Work *(12 minutes, with a partner)*

**1.** Compute Modified Duration from the Macaulay Duration above (Modified Duration = Macaulay Duration ÷ (1 + YTM)).

In [5]:
mod_duration = mac_duration / (1 + ytm)
print(f'Modified Duration = {mod_duration:.2f}')

Modified Duration = 2.62


**2.** Use Modified Duration to estimate the % price change and the new price if the yield rises from 8% to 9%.

In [6]:
delta_y = 0.01
est_pct_change = -mod_duration * delta_y
est_price = price * (1 + est_pct_change)
print(f'Estimated % change={est_pct_change:.2%}  Estimated new price=${est_price:.2f}')

Estimated % change=-2.62%  Estimated new price=$923.62


**3.** Reprice the bond exactly at 9% (same PV approach as Chapter 14) and compare it to your duration estimate. Which is higher, and what does that gap tell you about convexity?

In [7]:
new_ytm = ytm + delta_y
exact_price = (cash_flows / (1 + new_ytm) ** periods).sum()
convexity_gap = exact_price - est_price
print(f'Exact price=${exact_price:.2f}  Estimate=${est_price:.2f}  Gap (convexity)=${convexity_gap:.2f}')

Exact price=$924.06  Estimate=$923.62  Gap (convexity)=$0.44


## Step 4 — Python Pass *(7 minutes)*

- The Cash Flow / PV / t×PV table, with SUM() formulas feeding a Macaulay Duration cell
- Modified Duration and the %-price-change estimate, built from that cell
- An exact reprice at the new yield using PV(), placed right next to the duration estimate for comparison

In [8]:
def bond_duration(face, coupon_rate, n, ytm):
    periods = np.arange(1, n + 1)
    cfs = np.full(n, face * coupon_rate)
    cfs[-1] += face
    pv = cfs / (1 + ytm) ** periods
    price = pv.sum()
    mac = (periods * pv).sum() / price
    return {'price': price, 'macaulay': mac, 'modified': mac / (1 + ytm)}

bond_duration(1000, 0.06, 3, 0.08)

{'price': np.float64(948.4580602550423),
 'macaulay': np.float64(2.828615046736481),
 'modified': np.float64(2.619088006237482)}

## Step 5 — Discussion *(3-5 minutes, no calculation)*

> Two bonds have the same maturity, but Bond A pays a higher coupon than Bond B. Which bond has the higher duration, and why does that matter for someone trying to immunize a portfolio against interest-rate risk?